# 02 — Embeddings and Semantic Search

## Why this notebook exists

Notebook 01 left us with a precise problem: we need to send the model only the *relevant* documents, but to do that we have to *measure* relevance — and we hand-picked the right document ourselves, which is exactly the step that needs automating. Keyword matching is a weak proxy: a question about "how long the robot runs" should match a document that says "battery life," even though they share no words.

This notebook introduces the tool that solves it: **embeddings**. An embedding turns a piece of text into a vector of numbers, positioned so that text with similar *meaning* lands close together — even when the words differ. We'll look at what an embedding actually is, measure closeness with **cosine similarity** (computed by hand in NumPy, so nothing is hidden), watch semantic similarity beat keyword overlap, and then build a tiny search that ranks our Halcyon documents by relevance to a question. That ranking is the first real retrieval step — the automated version of notebook 01's hand-picking.

Requires an `OPENAI_API_KEY` (for the embeddings API). No vector database yet — just NumPy, so you can see exactly how it works.

## What you'll learn

- What an **embedding** is: a fixed-length vector of floats representing a piece of text, and how to get one from `text-embedding-3-small`.
- How to measure similarity between two embeddings with **cosine similarity**, written out in NumPy.
- Why **semantic** similarity beats **keyword** matching — a question and its answer can share almost no words yet still be close in embedding space.
- How to build a **brute-force top-k semantic search**: embed a corpus, embed a query, rank by cosine, return the best matches.
- Why embedding *whole documents* is only a starting point — and why notebook 03 needs to chunk them.

## 1. Setup

This notebook calls the OpenAI **embeddings** API and does all the math in NumPy. We load `OPENAI_API_KEY` from the environment (and from a local `.env` file if `python-dotenv` is installed — real environment variables always win), then define three small helpers:

- `embed(text)` — embed a single string into a NumPy vector.
- `embed_many(texts)` — embed a list of strings in one API call (cheaper and faster than looping).
- `cosine(a, b)` — cosine similarity between two vectors, a number in `[-1, 1]` where higher means more similar.

In [ ]:
import os

# Optional: load a local .env if python-dotenv is installed. Real env vars win.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ── Key guard ──────────────────────────────────────────────────────────────
if not os.environ.get("OPENAI_API_KEY"):
    print("=" * 60)
    print("OPENAI_API_KEY is not set.")
    print("=" * 60)
    print()
    print("This notebook calls the OpenAI embeddings API.")
    print()
    print("Set it and restart the kernel:")
    print("  export OPENAI_API_KEY=sk-...")
    raise SystemExit("Set OPENAI_API_KEY and restart the kernel to continue.")

print("OPENAI_API_KEY set ✓")

In [ ]:
import numpy as np
from openai import OpenAI

EMBED_MODEL = "text-embedding-3-small"

openai_client = OpenAI()  # reads OPENAI_API_KEY from the environment


def embed(text: str) -> np.ndarray:
    """Embed a single string into a NumPy vector."""
    resp = openai_client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.array(resp.data[0].embedding, dtype=np.float32)


def embed_many(texts: list) -> np.ndarray:
    """Embed a list of strings in ONE API call.

    Returns a matrix of shape (len(texts), embedding_dim); row i is the
    embedding of texts[i]. The API preserves input order.
    """
    resp = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in resp.data], dtype=np.float32)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two vectors: the cosine of the angle between
    them. 1.0 = identical direction, 0.0 = unrelated (orthogonal), -1.0 = opposite.
    """
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print("Setup OK")
print(f"Embedding model: {EMBED_MODEL}")

## 2. What Is an Embedding?

An embedding is a list of numbers — a vector — that represents a piece of text. The embedding model is trained so that texts with similar meaning produce vectors that point in similar directions. `text-embedding-3-small` returns a **1536-dimensional** vector for any input, from a single word to a full paragraph.

The cell below embeds a short phrase and inspects the result: how many dimensions it has, its data type, its length (norm), and its first few values. The exact numbers aren't meaningful on their own — what matters is that *every* piece of text becomes a vector of the same shape, so we can compare any two of them.

In [ ]:
vec = embed("warehouse robot")

print("Embedding of 'warehouse robot':")
print(f"  dimensions : {vec.shape[0]}")
print(f"  dtype      : {vec.dtype}")
print(f"  vector norm: {np.linalg.norm(vec):.4f}")
print(f"  first 8    : {np.round(vec[:8], 4)}")

## 3. Cosine Similarity by Hand

To compare two embeddings we measure the **angle** between their vectors with cosine similarity:

```
cosine(a, b) = (a · b) / (‖a‖ · ‖b‖)
```

It ranges from -1 to 1; for these embeddings, more-similar text gives a higher value. We defined `cosine` in NumPy in Section 1 — nothing is hidden.

The cell below scores four sentence pairs. Two pairs *mean* the same thing but use different words; two pairs *share* words but mean different things. Watch the meaning-based pairs score higher — that's the whole reason embeddings beat keyword matching.

> **Gotcha:** Cosine values from a real embedding model are not calibrated probabilities — an unrelated pair often scores 0.1–0.4, not 0.0. What matters is the *ranking*: the semantically-matched pair scores clearly higher than the lexically-overlapping-but-unrelated one. Compare scores to each other, not to an absolute threshold.

In [ ]:
pairs = [
    # Same meaning, few shared words → should score HIGH.
    ("How long does the robot run on one charge?", "battery life per charge"),
    # Shared words ("run", "long") but different meaning → should score LOWER.
    ("How long does the robot run on one charge?", "The run was long and the day tiring."),
    # Same meaning, different words → should score HIGH.
    ("I need to buy a new car.", "looking to purchase an automobile"),
    # No relationship → should score LOW.
    ("I need to buy a new car.", "The weather is sunny today."),
]

for a, b in pairs:
    sim = cosine(embed(a), embed(b))
    print(f"  cosine = {sim:.3f}")
    print(f"     A: {a}")
    print(f"     B: {b}")
    print()

## 4. Semantic Search Beats Keyword Search

Here is the concrete payoff. Take a real user question about the Porter P2 and the line in our corpus that answers it. They share almost no words — the question says "operate before recharging," the document says "battery life per charge." A keyword search would miss the connection entirely. Embeddings catch it.

The cell below computes a naive keyword-overlap score (what fraction of the question's words appear in the document) and the cosine similarity, side by side.

In [ ]:
query = "How long can the Porter P2 operate before it needs recharging?"
doc = "Battery life: 12 hours per charge"

# Naive keyword overlap: fraction of query words that appear in the document.
q_words = set(query.lower().replace("?", "").split())
d_words = set(doc.lower().split())
shared = q_words & d_words
overlap = len(shared) / len(q_words)

sim = cosine(embed(query), embed(doc))

print(f"Query : {query}")
print(f"Doc   : {doc}")
print()
print(f"Keyword overlap : {overlap:.2f}   shared words: {sorted(shared)}")
print(f"Cosine (meaning): {sim:.3f}")
print()
print("Keyword search barely connects them — they share essentially no words.")
print("Embeddings see that 'operate before recharging' means 'battery life per charge'.")

## 5. Brute-Force Semantic Search Over the Corpus

Now we automate notebook 01's hand-picking. We load the five Halcyon documents, embed each one (a single batched API call), and store the vectors in a matrix. To answer a question we embed the question, compute its cosine similarity against every document vector, and rank them. The top result is the document most likely to hold the answer — chosen automatically, no human in the loop.

This is "brute force" because we compare the query against *every* document. That's fine for five documents; notebook 04 introduces a real vector index for when there are thousands.

In [ ]:
from pathlib import Path

# Resolve the data folder whether the kernel runs from rag/ or from the repo root.
DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("rag/data")

doc_names: list = []
doc_texts: list = []
for path in sorted(DATA_DIR.glob("*.md")):
    doc_names.append(path.name)
    doc_texts.append(path.read_text(encoding="utf-8"))

# Fail clearly if the corpus is missing, instead of a confusing error later.
if not doc_texts:
    raise FileNotFoundError(
        f"No .md documents found in {DATA_DIR}/. "
        "Run notebook 01 first to create the Halcyon corpus."
    )

# Embed every document once, in a single batched call.
doc_matrix = embed_many(doc_texts)
print(f"Embedded {len(doc_texts)} documents into a {doc_matrix.shape} matrix:")
for name in doc_names:
    print(f"  - {name}")

In [ ]:
def semantic_search(query: str, top_k: int = 3) -> list:
    """Return the top_k (doc_name, score) pairs most similar to the query,
    ranked by cosine similarity against the pre-embedded document matrix.
    """
    q = embed(query)
    scored = [(doc_names[i], cosine(q, doc_matrix[i])) for i in range(len(doc_names))]
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return scored[:top_k]


queries = [
    "How long can the Porter P2 run before recharging?",
    "Who founded Halcyon and where is it based?",
    "What does the warranty cover?",
]

for query in queries:
    print(f"Q: {query}")
    for name, score in semantic_search(query, top_k=3):
        print(f"   {score:.3f}  {name}")
    print()

## 6. What We Built — and Where It Breaks

We just built real retrieval: given a question, rank the documents by semantic relevance and return the best ones. Feed the top result into the prompt instead of the whole corpus, and you get notebook 01's "it works" quality without notebook 01's waste. That is the core of RAG.

But there's a crack already. We embedded each document as a *single* vector. A real document isn't about one thing — `support.md` covers warranty, support hours, *and* spare-part depots. Squashing all of that into one vector blurs it: a question about spare parts and a question about warranty both match the same averaged vector, and a long document's specific details get washed out. Worse, real documents are far longer than ours and won't embed well as one piece at all.

The fix is to break documents into smaller, focused **chunks** and embed those — so retrieval can return the exact passage that answers a question, not a whole averaged document.

## What you just learned

- An **embedding** is a fixed-length vector (1536 dims for `text-embedding-3-small`) representing text by meaning; similar meanings point in similar directions.
- **Cosine similarity** measures the angle between two embeddings; compare scores to *each other*, not to an absolute threshold.
- **Semantic similarity beats keyword matching**: a question and its answer can share almost no words yet still be close in embedding space.
- A **brute-force semantic search** — embed the corpus, embed the query, rank by cosine — automatically picks the relevant document, replacing notebook 01's hand-picking.
- Embedding *whole documents* is too coarse: one vector blurs a multi-topic document, which is why we need chunking.

## What's missing

We embedded whole documents, and our documents are tiny. Real corpora are PDFs and long pages that cover many topics each — too big and too mixed to represent as a single vector. We also haven't dealt with *loading* anything beyond plain markdown.

**Notebook 03 — `03_chunking_and_document_loading.ipynb`** tackles both: loading real documents (including a PDF with `pypdf`), then splitting them into well-sized, overlapping **chunks** with metadata (source, page). Chunking is what lets retrieval return the exact passage that answers a question instead of a whole averaged document — and it's the input the vector index in notebook 04 will store.